# Performance Metrics Extraction — DWDM Demultiplexer

This notebook:
1. Runs the analytical spectral simulation for all 8 ITU C-band channels.
2. Extracts IL, ER, Q, FWHM, shape factor, FSR, and crosstalk.
3. Exports the results as a CSV (suitable for inclusion in the IEEE paper).
4. Generates publication-quality figures.

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
sys.path.insert(0, str(repo_root / 'design'))
sys.path.insert(0, str(repo_root / 'simulation'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import config as cfg
from spectral_response import simulate_all_channels, extract_metrics

print('Imports OK.')

## 1. Simulate All Channels

In [ ]:
results = simulate_all_channels(
    num_channels=cfg.NUM_CHANNELS,
    wl_start=1530.0,
    wl_stop=1565.0,
    n_points=3500,
    kappa_sq=0.08,
)
print(f'Simulated {len(results)} channels.')

## 2. Per-Channel Metrics Table

In [ ]:
rows = []
for k, data in results.items():
    m = data['metrics']
    rows.append({
        'Channel': f'Ch{k+1}',
        'ITU λ (nm)': data['target_nm'],
        'Peak λ (nm)': m['peak_wl_nm'],
        'Δλ (nm)': round(m['peak_wl_nm'] - data['target_nm'], 3),
        'IL (dB)': m['il_db'],
        'ER (dB)': m['er_db'],
        'FWHM (nm)': m['fwhm_nm'],
        'Q': m['q_factor'],
        'Shape factor η': m['shape_factor'],
        'FSR (nm)': m['fsr_nm'],
    })

df = pd.DataFrame(rows)
display(df)

In [ ]:
# Export to CSV for LaTeX table generation
csv_path = repo_root / 'simulation' / 'metrics.csv'
df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

## 3. Spectral Response — Publication Figure

In [ ]:
fig = plt.figure(figsize=(12, 6))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.4)

colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

ax_main = fig.add_subplot(gs[0, :])
for k, data in results.items():
    wl     = data['wl']
    T_drop = 10 * np.log10(np.clip(data['T_drop'], 1e-12, None))
    T_thru = 10 * np.log10(np.clip(data['T_thru'], 1e-12, None))
    ax_main.plot(wl, T_drop, color=colors[k], lw=1.5,
                 label=f'Ch{k+1} ({data["target_nm"]:.2f}nm)')
    ax_main.plot(wl, T_thru, color=colors[k], lw=0.7, ls='--')
    ax_main.axvline(data['target_nm'], color=colors[k], lw=0.5, ls=':', alpha=0.6)

ax_main.set_xlabel('Wavelength (nm)')
ax_main.set_ylabel('Transmission (dB)')
ax_main.set_title('8-Channel DWDM Demultiplexer — Spectral Response')
ax_main.set_ylim(-40, 2)
ax_main.legend(fontsize=7, ncol=4, loc='lower left')
ax_main.grid(True, alpha=0.3)

# Per-channel zoom insets
for k in range(4):
    data   = results[k]
    wl_c   = data['target_nm']
    wl_w   = 3.0   # ±1.5 nm window
    ax_sub = fig.add_subplot(gs[1, k])
    mask   = (data['wl'] >= wl_c - wl_w) & (data['wl'] <= wl_c + wl_w)
    T_drop = 10 * np.log10(np.clip(data['T_drop'][mask], 1e-12, None))
    ax_sub.plot(data['wl'][mask], T_drop, color=colors[k], lw=1.5)
    ax_sub.axvline(wl_c, color='k', lw=0.5, ls='--')
    ax_sub.set_title(f'Ch{k+1}\n{wl_c:.2f}nm', fontsize=8)
    ax_sub.set_xlabel('λ (nm)', fontsize=7)
    ax_sub.set_ylabel('dB', fontsize=7)
    ax_sub.tick_params(labelsize=7)
    ax_sub.grid(True, alpha=0.3)
    ax_sub.set_ylim(-35, 2)

fig_path = repo_root / 'latex' / 'figures' / 'spectra_8ch.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150)
plt.show()
print(f'Figure saved: {fig_path}')

## 4. Crosstalk Matrix

In [ ]:
n = len(results)
xtalk = np.zeros((n, n))

for i, di in results.items():
    peak_wl = di['metrics']['peak_wl_nm']
    for j, dj in results.items():
        if i == j:
            xtalk[i, j] = 0.0
        else:
            # Power of channel j's drop port at channel i's peak wavelength
            idx = np.argmin(np.abs(dj['wl'] - peak_wl))
            T_unwanted = dj['T_drop'][idx]
            T_desired  = di['T_drop'][np.argmax(di['T_drop'])]
            xtalk[i, j] = 10 * np.log10(max(T_unwanted / max(T_desired, 1e-12), 1e-12))

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(xtalk, cmap='RdYlGn', vmin=-60, vmax=0)
plt.colorbar(im, ax=ax, label='Crosstalk (dB)')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels([f'Ch{i+1}' for i in range(n)])
ax.set_yticklabels([f'Ch{i+1}' for i in range(n)])
ax.set_xlabel('Interfering channel')
ax.set_ylabel('Target channel drop port')
ax.set_title('Inter-Channel Crosstalk Matrix (dB)')

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{xtalk[i,j]:.0f}', ha='center', va='center',
                fontsize=7, color='black')

plt.tight_layout()
xt_path = repo_root / 'latex' / 'figures' / 'crosstalk_matrix.png'
plt.savefig(xt_path, dpi=150)
plt.show()
print(f'Figure saved: {xt_path}')